In [1]:
import pandas as pd
import numpy as np
import random
from faker import Faker
from datetime import datetime

fake = Faker("en_IN")

random.seed(42)
np.random.seed(42)
Faker.seed(42)

In [2]:
customers_df = pd.read_csv("Customers1.csv")

accounts_df = pd.read_csv("Accounts.csv")

In [4]:
customers_df["Join_Date"] = pd.to_datetime(customers_df["Join_Date"])

accounts_df["Opening_Date"] = pd.to_datetime(accounts_df["Opening_Date"])

In [5]:
merged_df = accounts_df.merge(
    customers_df,
    on="Customer_ID",
    how="left"
)

In [6]:
merged_df.shape

(15000, 21)

In [7]:
merged_df.head()

,Account_ID,Customer_ID,Account_Type,Branch_Code,Branch_City,Opening_Date,Current_Balance,Status,First_Name,Last_Name,...,DOB,Email,Phone,City,State,Occupation,Annual_Income,Marital_Status,Customer_Segment,Join_Date
0,A000000001,C00001,Savings,BR1007,Hyderabad,2026-02-18,508838,Active,Isaac,Bakshi,...,1972-10-13,isaac.bakshi719@email.com,8340505846,Bengaluru,Karnataka,Student,147127,Married,Retail,2024-11-23
1,A000000002,C00002,Current,BR1009,Ahmedabad,2022-02-15,1654416,Active,Anvi,Konda,...,1976-10-19,anvi.konda164@email.com,8998485882,Kolkata,West Bengal,Doctor,2697498,Married,VIP,2020-06-27
2,A000000003,C00002,Savings,BR1004,Delhi,2023-07-24,726488,Active,Anvi,Konda,...,1976-10-19,anvi.konda164@email.com,8998485882,Kolkata,West Bengal,Doctor,2697498,Married,VIP,2020-06-27
3,A000000004,C00003,Savings,BR1005,Lucknow,2026-04-11,1216997,Active,Liam,Chaudry,...,1988-05-26,liam.chaudry390@email.com,6415393687,Chennai,Tamil Nadu,Data Analyst,1300581,Single,Premium,2024-05-06
4,A000000005,C00004,Savings,BR1024,Lucknow,2019-09-10,230995,Active,Gagan,Sami,...,1982-10-10,gagan.sami550@email.com,6536124280,Kolkata,West Bengal,Doctor,2692670,Married,VIP,2018-11-10


In [8]:
MERCHANTS = {

    "Shopping": [
        "Amazon",
        "Flipkart",
        "Myntra"
    ],

    "Food": [
        "Swiggy",
        "Zomato",
        "Domino's"
    ],

    "Travel": [
        "Uber",
        "Ola",
        "IRCTC"
    ],

    "Bills": [
        "CESC",
        "Airtel",
        "Jio"
    ],

    "Healthcare": [
        "Apollo Pharmacy",
        "MedPlus",
        "Fortis"
    ],

    "Grocery": [
        "BigBasket",
        "DMart",
        "Reliance Fresh"
    ],

    "Fuel": [
        "Indian Oil",
        "BPCL",
        "HPCL"
    ],

    "Entertainment": [
        "Netflix",
        "BookMyShow",
        "Spotify"
    ],

    "Salary": [
        "Employer Payroll"
    ],

    "Investment": [
        "Groww",
        "Zerodha",
        "Upstox"
    ]
}

In [9]:
PAYMENT_MODES = [
    "UPI",
    "Debit Card",
    "Credit Card",
    "Net Banking",
    "Cash Deposit",
    "Cheque"
]

PAYMENT_WEIGHTS = [
    45,
    20,
    10,
    10,
    8,
    7
]

In [10]:
TRANSACTION_TYPES = [
    "Debit",
    "Credit"
]

TRANSACTION_WEIGHTS = [
    75,
    25
]

In [11]:
AMOUNT_RANGES = {

    "Shopping": (500, 50000),

    "Food": (100, 2000),

    "Travel": (500, 30000),

    "Bills": (300, 15000),

    "Healthcare": (500, 25000),

    "Grocery": (200, 5000),

    "Fuel": (500, 5000),

    "Entertainment": (200, 8000),

    "Salary": (20000, 250000),

    "Investment": (5000, 100000)

}

In [12]:
transaction_counts = np.random.randint(
    15,
    51,
    size=len(merged_df)
)

In [13]:
transaction_counts.min()

np.int32(15)

In [14]:
transaction_counts.max()

np.int32(50)

In [15]:
transaction_counts.mean()

np.float64(32.49373333333333)

In [16]:
transaction_counts.sum()

np.int64(487406)

In [17]:
transaction_counts.sum()

np.int64(487406)

In [18]:
transactions = []

transaction_number = 1

In [19]:
def generate_amount(category):

    low, high = AMOUNT_RANGES[category]

    return round(random.uniform(low, high), 2)

In [20]:
def get_merchant(category):

    return random.choice(MERCHANTS[category])

In [21]:
def choose_category(customer_segment):

    if customer_segment == "Retail":

        categories = [
            "Grocery",
            "Food",
            "Bills",
            "Fuel",
            "Shopping",
            "Entertainment"
        ]

        weights = [30,25,15,15,10,5]

    elif customer_segment == "Premium":

        categories = [
            "Shopping",
            "Travel",
            "Food",
            "Bills",
            "Investment",
            "Entertainment",
            "Healthcare"
        ]

        weights = [25,20,20,10,10,10,5]

    else:

        categories = [
            "Shopping",
            "Travel",
            "Investment",
            "Healthcare",
            "Entertainment",
            "Food"
        ]

        weights = [30,25,20,10,10,5]

    return random.choices(
        categories,
        weights=weights,
        k=1
    )[0]

In [27]:
transactions = []
transaction_number = 1

In [28]:
for index, row in merged_df.iterrows():

    account_id = row["Account_ID"]
    customer_id = row["Customer_ID"]

    customer_segment = row["Customer_Segment"]

    city = row["City"]

    opening_date = row["Opening_Date"]

    number_of_transactions = transaction_counts[index]

    for _ in range(number_of_transactions):

        category = choose_category(customer_segment)

        merchant = get_merchant(category)

        amount = generate_amount(category)

        payment_mode = random.choices(
            PAYMENT_MODES,
            weights=PAYMENT_WEIGHTS,
            k=1
        )[0]

        transaction_type = random.choices(
            TRANSACTION_TYPES,
            weights=TRANSACTION_WEIGHTS,
            k=1
        )[0]

        transaction_date = fake.date_time_between(
            start_date=opening_date,
            end_date="now"
        )

        transaction_id = f"T{transaction_number:09d}"

        transaction_number += 1

        transactions.append({

            "Transaction_ID": transaction_id,

            "Account_ID": account_id,

            "Customer_ID": customer_id,

            "Transaction_Date": transaction_date,

            "Merchant": merchant,

            "Merchant_Category": category,

            "Amount": amount,

            "Payment_Mode": payment_mode,

            "Transaction_Type": transaction_type,

            "City": city

        })

In [29]:
len(transactions)

487406

In [30]:
transactions_df = pd.DataFrame(transactions)

transactions_df.shape

(487406, 10)

In [31]:
transactions_df.head()

,Transaction_ID,Account_ID,Customer_ID,Transaction_Date,Merchant,Merchant_Category,Amount,Payment_Mode,Transaction_Type,City
0,T000000001,A000000001,C00001,2026-06-22 03:56:41,CESC,Bills,8151.86,UPI,Debit,Bengaluru
1,T000000002,A000000001,C00001,2026-03-11 14:50:25,Reliance Fresh,Grocery,3503.18,UPI,Debit,Bengaluru
2,T000000003,A000000001,C00001,2026-02-22 20:33:30,Indian Oil,Fuel,1530.72,UPI,Debit,Bengaluru
3,T000000004,A000000001,C00001,2026-07-11 23:52:32,BigBasket,Grocery,4583.05,Debit Card,Debit,Bengaluru
4,T000000005,A000000001,C00001,2026-04-12 09:43:46,DMart,Grocery,2099.03,Cash Deposit,Debit,Bengaluru


In [32]:
transactions_df["Transaction_ID"].duplicated().sum()

np.int64(0)

In [33]:
transactions_df.isnull().sum()

Transaction_ID       0
Account_ID           0
Customer_ID          0
Transaction_Date     0
Merchant             0
Merchant_Category    0
Amount               0
Payment_Mode         0
Transaction_Type     0
City                 0
dtype: int64

In [34]:
transactions_df["Transaction_Type"].value_counts(normalize=True) * 100

Transaction_Type
Debit     75.075399
Credit    24.924601
Name: proportion, dtype: float64

In [35]:
transactions_df["Payment_Mode"].value_counts(normalize=True) * 100

Payment_Mode
UPI             45.061407
Debit Card      19.961182
Credit Card     10.045834
Net Banking      9.956381
Cash Deposit     8.016110
Cheque           6.959085
Name: proportion, dtype: float64

In [36]:
transactions_df["Merchant_Category"].value_counts()

Merchant_Category
Shopping         104584
Food              91812
Travel            73501
Bills             47163
Investment        43249
Grocery           42682
Entertainment     41336
Healthcare        21855
Fuel              21224
Name: count, dtype: int64

In [37]:
transactions_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 487406 entries, 0 to 487405
Data columns (total 10 columns):
 #   Column             Non-Null Count   Dtype         
---  ------             --------------   -----         
 0   Transaction_ID     487406 non-null  object        
 1   Account_ID         487406 non-null  object        
 2   Customer_ID        487406 non-null  object        
 3   Transaction_Date   487406 non-null  datetime64[ns]
 4   Merchant           487406 non-null  object        
 5   Merchant_Category  487406 non-null  object        
 6   Amount             487406 non-null  float64       
 7   Payment_Mode       487406 non-null  object        
 8   Transaction_Type   487406 non-null  object        
 9   City               487406 non-null  object        
dtypes: datetime64[ns](1), float64(1), object(8)
memory usage: 37.2+ MB


In [38]:
transactions_df.to_csv(
    "Transactions.csv",
    index=False
)